# Homework 2 Worked Example: Forecasting Corn Real Prices

### AGEC 370: Agricultural Price Analysis

Click the badge below to open in Google Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chuckgrigsby0/agec_370/blob/main/code/notebooks/hw02_example_corn_price_forecasting.ipynb)

---

## Objective

Forecast the U.S. real corn price (inflation-adjusted) one to three years ahead. Compare five simple forecast methods: the mean, the last value, moving average, a linear trend regression model, and an autoregression model, AR(1), which uses last year's price to predict this year's price. 

Evaluate each approach using the mean-squared error (MSE) and root mean-squared error (RSE) using the holdout data. 

---

## Assignment Instructions

1. Save a copy of this notebook to your Google Drive.
2. Run every cell in order from top to bottom.
3. Read the comments above each step before running the code. Review the output after running code cells.
4. Open the [Homework 2 notebook](https://colab.research.google.com/github/chuckgrigsby0/agec_370/blob/main/code/notebooks/hw02_hog_price_forecasting.ipynb). Complete the same steps with the hog price data.

## Data Description

The file `corn_production_price.csv` has one row per year, 1913 to 2025. Production is for the calendar year. The price is the marketing year average, labeled by the year the marketing year starts (September to August). Real prices are calculated below, as in Homework 1.

| Column | Description | Units |
|---|---|---|
| `year` | Calendar year for production. Marketing year for price. | |
| `production_billion_bu` | U.S. corn production (USDA NASS) | Billion bushels |
| `nominal_price` | Marketing year average price received by U.S. corn producers (USDA NASS) | Dollars per bushel |
| `annual_avg_cpi` | Consumer Price Index for All Urban Consumers (CPI-U), all items, annual average (BLS) | Index (1982-84 = 100) |

## Package and Data Loading

Import the libraries, load the data, and calculate real prices. This repeats steps from Homework 1. 

In [ ]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import the regression library used for the trend and AR(1) forecasts
# 'smf' lets us write a regression as a formula, such as 'outcome ~ explanatory variable'
import statsmodels.formula.api as smf

In [ ]:
# Load data from GitHub repository
base_url = "https://raw.githubusercontent.com/chuckgrigsby0/agec_370/main/data/hw/"
corn = pd.read_csv(base_url + "corn_production_price.csv")

# Real price in 2025 dollars: real_price = (nominal_price / CPI_t) * CPI_2025
cpi_base_year = corn[corn['year'] == 2025]['annual_avg_cpi'].values[0]
corn['real_price'] = (corn['nominal_price'] / corn['annual_avg_cpi']) * cpi_base_year

corn[['year', 'nominal_price', 'real_price']].tail()

### Plot the Real Price

We will build forecasts for real corn prices. Visualizing our data first can help us analyze important trends or anomalies in the data. 

In [ ]:
sns.set_style('whitegrid')

plt.figure(figsize=(14, 6))  # Width and height in inches

sns.lineplot(
    data=corn,            # DataFrame containing the corn data
    x='year',             # Plot year on the x-axis
    y='real_price',   # Plot the real price on the y-axis
    linewidth=1.2,        # Increase for thicker lines
    color='#FF8200'       # Line color
)

plt.title("Annual Real Corn Prices ($/bushel, 2025 Dollars)", fontsize=14, weight='bold')
plt.xlabel("Year", fontsize=14, weight='bold')

plt.ylabel("Real Price ($/bushel, 2025 Dollars)", fontsize=14, weight='bold')

plt.xticks(rotation=90, ha='center', fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

## Splitting the Data by Time

We will evaluate the predictive accuracy of our forecasts using data not used to fit the model. 

We will create a holdout, test dataset by splitting the data by time. We will use earlier years to train the predictive model (the *training* data) and later years to evaluate the trained model's accuracy (the *test* data).

In this example, we will use 2019 as the cutoff for model training. Our test data will range from 2020 to 2025.

In [ ]:
holdout_year = 2019

# Training years: keep the rows with a year at or before the cutoff
train = corn[corn['year'] <= holdout_year].copy()

# Test years: keep the rows with a year after the cutoff
test = corn[corn['year'] > holdout_year].copy()

print(f"Training years: {train['year'].min()} to {train['year'].max()} ({len(train)} years)")
print(f"Test years: {test['year'].min()} to {test['year'].max()} ({len(test)} years)")

## Baseline Forecasts

Three simple forecasts. For each test year, the forecast uses only information from earlier years.

| Method | Forecast for year *t* |
|---|---|
| Mean | Average real price during the training years |
| Last value | Real price in year *t* - 1 |
| 3-year moving average | Average real price during years *t* - 3, *t* - 2, and *t* - 1 |

A *lag* is the same series shifted back in time: last year's price is the one-year lag. The last value and moving average forecasts use actual prices from the years just before each test year, so they are *one-year-ahead* forecasts. The mean uses only the training years.

In [ ]:
# Mean forecast: each forecast is the mean of the training data
train_mean = train['real_price'].mean()
corn['forecast_mean'] = train_mean

In [ ]:
# Last value forecast: move each value down one row, so each row holds the previous year's price.
# We store last year's price as real_price_lag1. The AR(1) model in the next section also uses it.
corn['real_price_lag1'] = corn['real_price'].shift(1)
corn['forecast_last'] = corn['real_price_lag1']

In [ ]:
# 3-year moving average forecast: average each year with the two years before it,
# then move the result down one row, so the forecast for year t uses years t - 3 to t - 1.
corn['forecast_ma3'] = corn['real_price'].rolling(3).mean().shift(1)

In [ ]:
# The train and test data were copied before the forecast columns existed.
# We need to copy them again from the full table so they include the new columns.
train = corn[corn['year'] <= holdout_year].copy()
test = corn[corn['year'] > holdout_year].copy()

test[['year', 'real_price', 'forecast_mean', 'forecast_last', 'forecast_ma3']].round(2)

If you rerun the cell above, rerun the Regression Forecasts cells below it too. Copying the tables again removes the forecast columns they add.

In [ ]:
# Check: this cell runs without an error when the baseline forecasts use only earlier years
assert np.isclose(train_mean, train['real_price'].mean()), "The mean forecast should use the training years only."
assert np.isclose(test['forecast_last'].iloc[0], train['real_price'].iloc[-1]), "The last value forecast should look back one year, not forward."
assert np.isclose(test['forecast_ma3'].iloc[0], train['real_price'].iloc[-3:].mean()), "The moving average should average the three years before the forecast year."
print("Baseline forecasts use only earlier years.")

## Regression Forecasts

Two of our forecasts use regression. Regression fits a straight line through our data points (the line yielding smallest squared error). `.fit()` estimates predicted prices during the training years, while `.predict()` applies the forecast model to test data. 

- **OLS (ordinary least squares) linear trend:** `real_price = b0 + b1 * year`. The slope `b1` is the average change in the real price per year.
- **AR(1):** `real_price = a + b * real_price_lag1`. The coefficient `b` measures how closely this year's price is associated with last year's price. A value near 1 means prices persist over time. A value near 0 means last year's price has minimal explanatory power for this year's price. 

In `smf.ols()`, the formula `'outcome ~ explanatory variable'` names columns. The first row of the training data has no lagged price, so the regression drops it automatically.

In [ ]:
# OLS trend: regress the real price on the year, using the training years only.
trend_model = smf.ols('real_price ~ year', data=train).fit()

trend_slope = trend_model.params['year']  # estimated change in real price per year
print(f"Trend slope: {trend_slope:.2f} ($/bushel per year)")

# predict() applies the fitted line to the years in the test table
test['forecast_trend'] = trend_model.predict(test)

In [ ]:
# AR(1): regress the real price on last year's real price, using the training years only.
ar1_model = smf.ols('real_price ~ real_price_lag1', data=train).fit()

ar1_coef = ar1_model.params['real_price_lag1']  # persistence from one year to the next
print(f"AR(1) coefficient: {ar1_coef:.3f}")

test['forecast_ar1'] = ar1_model.predict(test)

## Forecast Accuracy

The *forecast error* each year is computed by subtracting the predicted price from the actual price. Prediction errors can be positive or negative, so we square them. We then average the squared errors to compute the MSE. When we take the square root of the MSE, we obtain the RMSE, which converts our errors to the same units as the price (dollars per bushel). A smaller RMSE indicates a more accurate forecast.

`RMSE = sqrt( mean( (actual - forecast)^2 ) )`

Based on the patterns in the data, which prediction method do you expect to perform best and worst? 

We will create a function to efficiently comput the RMSE. A *function* packages steps under a name. `rmse(actual, forecast)` below takes two arguments (`actual` and `forecast`) and returns the RMSE.

In [ ]:
def rmse(actual, forecast):
    error = actual - forecast  # forecast error in each test year
    # The next line squares each error, takes the mean, then takes the square root
    return np.sqrt((error ** 2).mean())

In [ ]:
# Calculate the RMSE of each method over the test years
methods = {
    'Mean': 'forecast_mean',
    'Last value': 'forecast_last',
    '3-year moving average': 'forecast_ma3',
    'OLS trend': 'forecast_trend',
    'AR(1)': 'forecast_ar1',
}
# This line applies rmse() to each method in turn and stores the results in one table
errors = pd.Series({name: rmse(test['real_price'], test[column]) for name, column in methods.items()})

best_method = errors.idxmin()

print(errors.round(2))
print(f"Smallest error: {best_method}")

An RMSE is in the units of the price. Dividing it by the average test-year price gives a rough sense of the error's size.

In [ ]:
# RMSE as a share of the average real price in the test years
(errors / test['real_price'].mean()).round(2)

### Compare the Forecasts to the Actual Price

The vertical line marks the cutoff. Lines to its right are forecasts for the test years. Gray lines are the three simple methods that need no fitted slope.

In [ ]:
plt.figure(figsize=(14, 6))

recent = corn[corn['year'] >= 2000]  # recent years only, so the test years are easy to read

sns.lineplot(data=recent, x='year', y='real_price', color='#0b2341', linewidth=2.5, label='Actual')
sns.lineplot(data=test, x='year', y='forecast_mean', color='#8a8f98', linestyle=':', linewidth=1.8, label='Mean')
sns.lineplot(data=test, x='year', y='forecast_last', color='#8a8f98', linestyle='--', linewidth=1.8, label='Last value')
sns.lineplot(data=test, x='year', y='forecast_ma3', color='#8a8f98', linestyle='-.', linewidth=1.8, label='3-year moving average')
sns.lineplot(data=test, x='year', y='forecast_trend', color='#FF8200', linestyle='--', linewidth=2, label='OLS trend')
sns.lineplot(data=test, x='year', y='forecast_ar1', color='#FF8200', linestyle='-', linewidth=2, label='AR(1)')

plt.axvline(x=holdout_year, color='black', linewidth=1)

plt.title("Actual and Forecast Real Corn Prices, Test Years 2020 to 2025", fontsize=14, weight='bold')
plt.xlabel("Year", fontsize=14, weight='bold')

plt.ylabel("Real Price ($/bushel, 2025 Dollars)", fontsize=14, weight='bold')

plt.xticks(rotation=90, ha='center', fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Forecast Horizon

The test above forecasts one year ahead and evaluates the methods on only six years. Planning decisions often look two or three years out.

The function below repeats the exercise many times. Each year from 2005 onward is a *forecast origin* (the last year of data a forecaster would know), as long as the data still include the year being forecast. For each origin, the function fits each method on the years up to the origin and forecasts `horizon` years later. Averaging over all origins gives about 20 forecasts instead of six.

For example, with origin 2010 and a horizon of 2, each method is fit on the years through 2010 and forecasts 2012. For AR(1), a forecast more than one year ahead uses its own earlier forecast as the next "last year" price.

In the function below, you only need to change the horizon.

Do you expect for the error of each method to rise, fall, or stay the same as the horizon grows from one year to three years?

In [ ]:
def rolling_rmse(data, horizon, first_origin=2005):
    """RMSE of each method, forecasting `horizon` years ahead from every origin year.

    Needs the columns year, real_price, and real_price_lag1.
    """
    data = data.sort_values('year').reset_index(drop=True)
    last_year = data['year'].max()
    # Create a dictionary to hold the squared errors for each method
    squared_errors = {'Mean': [], 'Last value': [], '3-year moving average': [], 'OLS trend': [], 'AR(1)': []}

    # An origin is the last year of data a forecaster would know
    for origin in range(first_origin, last_year - horizon + 1):
        known = data[data['year'] <= origin]  # only years up to the origin
        target_year = origin + horizon
        actual = data.loc[data['year'] == target_year, 'real_price'].iloc[0]

        # Re-fit both regressions using only the known years
        trend = smf.ols('real_price ~ year', data=known).fit()
        ar1 = smf.ols('real_price ~ real_price_lag1', data=known).fit()

        # AR(1) forecasts more than one year ahead feed each forecast back in as the next 'last year'
        forecast_ar1 = known['real_price'].iloc[-1]
        for step in range(horizon):
            forecast_ar1 = ar1.params['Intercept'] + ar1.params['real_price_lag1'] * forecast_ar1

        forecasts = {
            'Mean': known['real_price'].mean(),
            'Last value': known['real_price'].iloc[-1],
            '3-year moving average': known['real_price'].iloc[-3:].mean(),
            'OLS trend': trend.params['Intercept'] + trend.params['year'] * target_year,
            'AR(1)': forecast_ar1,
        }
        for name, forecast in forecasts.items():
            squared_errors[name].append((actual - forecast) ** 2)

    return pd.Series({name: np.sqrt(np.mean(values)) for name, values in squared_errors.items()})

In [ ]:
errors_1yr = rolling_rmse(corn, horizon=1)
errors_2yr = rolling_rmse(corn, horizon=2)
errors_3yr = rolling_rmse(corn, horizon=3)

horizon_table = pd.DataFrame({
    '1 year ahead': errors_1yr,
    '2 years ahead': errors_2yr,
    '3 years ahead': errors_3yr,
}).round(2)
horizon_table

In [ ]:
# Pull the AR(1) row from the table.
ar1_error_1yr = horizon_table.loc['AR(1)', '1 year ahead']

ar1_error_3yr = horizon_table.loc['AR(1)', '3 years ahead']

print(f"AR(1) error, 1 year ahead: {ar1_error_1yr}")
print(f"AR(1) error, 3 years ahead: {ar1_error_3yr}")

### Summary of Answers

Run the cell below to print the answers in one table.

In [ ]:
# Collect the answers computed above in one table
summary = pd.Series({
    'Training years': len(train),
    'Test years': len(test),
    'Training mean ($/bushel)': round(train_mean, 2),
    'Last value forecast, 2020 ($/bushel)': round(test.loc[test['year'] == 2020, 'forecast_last'].iloc[0], 2),
    '3-year average forecast, 2020 ($/bushel)': round(test.loc[test['year'] == 2020, 'forecast_ma3'].iloc[0], 2),
    'Trend slope ($/bushel per year)': round(trend_slope, 2),
    'AR(1) coefficient': round(ar1_coef, 3),
    'RMSE, Mean': round(errors['Mean'], 2),
    'RMSE, Last value': round(errors['Last value'], 2),
    'RMSE, 3-year moving average': round(errors['3-year moving average'], 2),
    'RMSE, OLS trend': round(errors['OLS trend'], 2),
    'RMSE, AR(1)': round(errors['AR(1)'], 2),
    'Method with the smallest RMSE': best_method,
    'AR(1) RMSE, 1 year ahead': ar1_error_1yr,
    'AR(1) RMSE, 3 years ahead': ar1_error_3yr,
}, name='Answer', dtype=object)  # object type keeps whole numbers as whole numbers
summary.to_frame()

## Next Step

Open the [Homework 2 notebook](https://colab.research.google.com/github/chuckgrigsby0/agec_370/blob/main/code/notebooks/hw02_hog_price_forecasting.ipynb) and complete the same steps with the hog price data.